In [1]:
# import pandas as pd
import dask.dataframe as dd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, RocCurveDisplay, roc_auc_score, precision_score, recall_score, f1_score
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

In [3]:
# Example with sample data
df = dd.read_csv('data/Friday.csv', assume_missing=True).compute()

In [4]:
model_columns = ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
       'Total Length of Fwd Packet', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags',
       'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Min',
       'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'CWR Flag Count',
       'ECE Flag Count', 'Down/Up Ratio', 'Fwd Bytes/Bulk Avg',
       'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg',
       'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Bwd Packets',
       'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'Fwd Act Data Pkts',
       'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max',
       'Active Min', 'Idle Std', 'ICMP Code', 'ICMP Type',
       'Total TCP Flow Time', 'Attempted Category']
model_target = 'Label'

In [5]:
data_columns = df.drop('Label', axis=1).columns.tolist()
data_target = 'Label'
data_columns

['id',
 'Flow ID',
 'Src IP',
 'Src Port',
 'Dst IP',
 'Dst Port',
 'Protocol',
 'Timestamp',
 'Flow Duration',
 'Total Fwd Packet',
 'Total Bwd packets',
 'Total Length of Fwd Packet',
 'Total Length of Bwd Packet',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd RST Flags',
 'Bwd RST Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Packet Length Min',
 'Packet Length Max',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance'

In [6]:
columns_to_use = []

for i in range(len(model_columns)):
    for j in range(len(data_columns)):
        if model_columns[i] == data_columns[j]:
            if model_columns[i] not in columns_to_use:
                columns_to_use.append(model_columns[i])
                print(model_columns[i], end=" ")
                break
columns_to_use

Src Port Dst Port Protocol Flow Duration Total Fwd Packet Total Length of Fwd Packet Fwd Packet Length Max Fwd Packet Length Min Fwd Packet Length Mean Fwd Packet Length Std Bwd Packet Length Max Bwd Packet Length Min Flow Bytes/s Flow Packets/s Flow IAT Mean Flow IAT Std Flow IAT Max Flow IAT Min Fwd IAT Mean Fwd IAT Std Bwd IAT Mean Bwd IAT Std Fwd PSH Flags Bwd PSH Flags Fwd URG Flags Bwd URG Flags Fwd RST Flags Bwd RST Flags Fwd Header Length Bwd Header Length Bwd Packets/s Packet Length Min FIN Flag Count SYN Flag Count RST Flag Count CWR Flag Count ECE Flag Count Down/Up Ratio Fwd Bytes/Bulk Avg Fwd Packet/Bulk Avg Fwd Bulk Rate Avg Bwd Bytes/Bulk Avg Bwd Bulk Rate Avg Subflow Fwd Packets Subflow Bwd Packets FWD Init Win Bytes Bwd Init Win Bytes Fwd Act Data Pkts Fwd Seg Size Min Active Mean Active Std Active Max Active Min Idle Std ICMP Code ICMP Type Total TCP Flow Time Attempted Category 

['Src Port',
 'Dst Port',
 'Protocol',
 'Flow Duration',
 'Total Fwd Packet',
 'Total Length of Fwd Packet',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd RST Flags',
 'Bwd RST Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Bwd Packets/s',
 'Packet Length Min',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'CWR Flag Count',
 'ECE Flag Count',
 'Down/Up Ratio',
 'Fwd Bytes/Bulk Avg',
 'Fwd Packet/Bulk Avg',
 'Fwd Bulk Rate Avg',
 'Bwd Bytes/Bulk Avg',
 'Bwd Bulk Rate Avg',
 'Subflow Fwd Packets',
 'Subflow Bwd Packets',
 'FWD Init Win Bytes',
 'Bwd Init Win Bytes',
 'Fwd Act Data Pkts',
 'Fwd Seg Size Min',
 'Activ

# Load model and test 

In [7]:
# define new model with basic parameters and load state dict weights
loaded_clf = TabNetClassifier()
loaded_clf.load_model("model.zip")

c:\Users\YOGA\Desktop\master s4 PFE\project\IDS-MAS\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [8]:
# Separate features and target
X = df[columns_to_use]
y = df[model_target]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")
# Check for categorical columns
categorical_columns = X.select_dtypes(include=['object']).columns
print(f"Categorical columns: {categorical_columns.tolist()}")

# Encode categorical variables if any
for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    print(f"Encoded {col}")

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nClasses: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")
# Display class distribution after encoding
unique, counts = np.unique(y_encoded, return_counts=True)
for i, (class_idx, count) in enumerate(zip(unique, counts)):
    print(f"Class {class_idx} ({label_encoder.classes_[class_idx]}): {count} samples")
X = X.dropna()

Features shape: (547557, 58)
Target shape: (547557,)
Feature columns: ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Length of Fwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Bwd IAT Mean', 'Bwd IAT Std', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Min', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg', 'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Bwd Packets', 'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'Fwd Act Data Pkts', 'Fwd 

In [9]:
# Test with one row to isolate the issue
try:
    single_pred = loaded_clf.predict(X.iloc[[0]].values)
    print("Single row prediction worked")
except Exception as e:
    print(f"Error on single row: {e}")

Single row prediction worked


In [10]:
single_pred

array([0])

In [11]:
loaded_clf# Get local (instance-level) feature importances
explain_matrix, masks = loaded_clf.explain(X.values)

# explain_matrix: shape (n_samples, n_features)
# Each row shows feature importance for that prediction

In [12]:
# Make predictions
y_pred = loaded_clf.predict(X.values)
y_pred_proba = loaded_clf.predict_proba(X.values)

In [16]:
# Calculate metrics
accuracy = accuracy_score(y_encoded, y_pred)
precision_weighted = precision_score(y_encoded, y_pred, average='weighted')
recall_weighted = recall_score(y_encoded, y_pred, average='weighted')
f1_weighted = f1_score(y_encoded, y_pred, average='weighted')

c:\Users\YOGA\Desktop\master s4 PFE\project\IDS-MAS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [19]:
# Multi-class AUC
# auc = roc_auc_score(y_encoded, y_pred_proba, multi_class='ovr', average='weighted')

# Results summary
results = {
    "Accuracy": f"{accuracy:.4f}",
    "Precision": f"{precision_weighted:.4f}",
    "Recall": f"{recall_weighted:.4f}",
    "F1-Score": f"{f1_weighted:.4f}",
    # "AUC": f"{auc:.4f}"
}


In [20]:
print("Model Performance:")
for metric, value in results.items():
    print(f"  {metric}: {value}")


Model Performance:
  Accuracy: 0.7013
  Precision: 0.7334
  Recall: 0.7013
  F1-Score: 0.7049


### Model Performance  

In [23]:
for metric, value in results.items():
    print(f"  {metric}: {value}")

# Compare against success criteria
print(f"\nSuccess Criteria Assessment:")
criteria_check = {
    "Accuracy > 95%": accuracy > 0.95,
    "Precision > 90%": precision_weighted > 0.90,
    "Recall > 85%": recall_weighted > 0.85,
    "F1-Score > 90%": f1_weighted > 0.90
}

for criterion, passed in criteria_check.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {criterion}: {status}")


  Accuracy: 0.7013
  Precision: 0.7334
  Recall: 0.7013
  F1-Score: 0.7049

Success Criteria Assessment:
  Accuracy > 95%: ✗ FAIL
  Precision > 90%: ✗ FAIL
  Recall > 85%: ✗ FAIL
  F1-Score > 90%: ✗ FAIL


In [ ]:
# save tabnet model
saving_path_name = "./tabnet_model_test_1"
saved_filepath = clf.save_model(saving_path_name)

# define new model with basic parameters and load state dict weights
loaded_clf = TabNetClassifier()
loaded_clf.load_model(saved_filepath)